# CONNECTING TO DATABASE

In [6]:
import redshift_connector

# Create a connection to the Redshift database with the above credentials
conn = redshift_connector.connect(
    host='c23-workgroup.129033205317.eu-west-2.redshift-serverless.amazonaws.com',
    database='dw_air_travel',
    user='yusuf_ahmed',
    password='Dyusuf_71',
    port=5439
)
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct laser_colour) FROM laser_incident')

# Extract the result from the cursor object
output = cursor.fetch_dataframe()

print(output)

# Close the cursor
cursor.close()

   count
0     84


### How many Distinct laser colours are there? 84

In [7]:
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct laser_colour) FROM laser_incident')
# Extract the result from the cursor object
output = cursor.fetch_dataframe()
print(output)
# Close the cursor
cursor.close()

   count
0     84


### How many distinct simplified laser colours? 71

In [8]:
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct (Lower(laser_colour))) FROM laser_incident')
# Extract the result from the cursor object
output = cursor.fetch_dataframe()
print(output)
# Close the cursor
cursor.close()

   count
0     71


## TASK 2: Extracting Data Into Pandas

In [18]:
import pandas as pd
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT *
            FROM laser_incident
            JOIN airport ON laser_incident.airport_id = airport.airport_id
            JOIN city ON airport.city_id = city.city_id
            JOIN state ON city.state_id = state.state_id
            WHERE state_name = 'Michigan'
        """)

        michigan_laser = cursor.fetch_dataframe()
        print("Number of rows:", len(michigan_laser))

    except Exception as e:
        conn.rollback()
        print(e)

Number of rows: 418


## Task 3
### How many rows are in the dataframe? 418
### What percentage of laser incidents cause significant harm? 0.72%
### What airport has the worst problem with laser incidents? DETROIT METRO WAYNE COUNTY = 176
### How has the frequency of laser incidents changed over time? Decreased slightly over time
### What are the most and least common laser colours used? Green as the Most, Orange as the least

In [20]:
injury = michigan_laser["injury"].value_counts()
print(injury)

injury
f    415
t      3
Name: count, dtype: int64


In [27]:
airport_counts = michigan_laser['airport_name'].value_counts()
print(airport_counts)
michigan_laser.info

airport_name
DETROIT METRO WAYNE COUNTY          176
KALAMAZOO/BATTLE CREEK INTL          37
GERALD R FORD INTL                   35
CAPITAL REGION INTL                  24
BISHOP INTL                          19
COLEMAN A YOUNG MUNI                 17
MBS INTL                             17
OAKLAND COUNTY INTL                  15
CHERRY CAPITAL                       12
MUSKEGON COUNTY                      10
JACKSON COUNTY/REYNOLDS FLD           5
BATTLE CREEK EXEC AT KELLOGG FLD      5
WILLOW RUN                            4
DELTA COUNTY                          4
SELFRIDGE ANGB                        3
MOUNT PLEASANT MUNI                   2
ST CLAIR COUNTY INTL                  2
HILLSDALE MUNI                        2
ALPENA COUNTY RGNL                    2
KIRSCH MUNI                           2
PADGHAM FLD                           2
OTTAWA EXEC                           2
CLARE COUNTY                          1
SOUTHWEST MICHIGAN RGNL               1
MAPLE GROVE                

<bound method DataFrame.info of      laser_incident_id flight_id aircraft  altitude laser_colour injury  \
0                15134   FDX1418     DC10      4000        Green      f   
1                 8224   FDX1269     A306      4000      Unknown      f   
2                27060   DAL1766     MD88      5000        Green      f   
3                 3673    DAL750     A319      1700        Green      f   
4                27820   DAL1433     MD88      9000       Purple      f   
..                 ...       ...      ...       ...          ...    ...   
413               4683   DAL1132     A319      8000        Green      f   
414               2475   CJT1316     B767     37000         Blue      f   
415              28778   PCL3822     CRJ9      5000        Green      f   
416              12043    RPA559   E170/L     28000       Orange      f   
417              16367   EDV5040     CRJ9      7000        Green      f   

     airport_id                  at  airport_id                airp

In [38]:
import altair as alt
alt.data_transformers.enable("vegafusion")

michigan_laser = michigan_laser.loc[:, ~michigan_laser.columns.duplicated()]

chart_data = (
    michigan_laser
    .assign(year=michigan_laser['at'].dt.to_period('Y').astype(str))
)

alt.Chart(chart_data).mark_line(point=True).encode(
    x=alt.X('year', title='Year', sort='x'),
    y=alt.Y('count()', title='Total Incidents'),
    tooltip=[
        alt.Tooltip('year', title='Year'),
        alt.Tooltip('count()', title='Total Incidents')
    ]
).properties(
    width=800,
    height=400,
    title='Incidents over time'
)

alt.Chart(...)

In [39]:
alt.Chart(michigan_laser).mark_bar().encode(
    x=alt.X("laser_colour", title="Colour of Laser", sort="-y"),
    y=alt.Y("count()", title="Count"),
    color=alt.Color('laser_colour')
).properties(
    title="Different laser colours",
    width=600,
    height=400)

alt.Chart(...)